# Nurse scheduling problem

This is a formulation to following nurse scheduling problem:

In a hospital, somewhere in Tunisia, patients with specific health conditions must get a treatment that requires the use of certain machines. Treatments are administrated to each patient in daily sessions (one session per day uninterruptedly), and also if the patient starts the treatment in a day then the other sessions must occur in the following days. For example, if Bob needs four sessions and he starts treatment on May 3rd, then his sessions must take place on May 3rd, 4th, 5th, and 6th.

The durations of sessions are determined by the doctors but it may vary from session to session for different patients. For example, Bob's first session may take one hour, the second may take 30 minutes, and so on. To simplify (and generalize) our notation, let's mesure session durations in slots, all slots having a common predefined duration, let say 30 minutes. Then, in Bob's example, we can say that the durations of the first two sessions are 2 slots and 1 slot, respectively. Let's also assume that the duration of every session is an integer number of slots.

Each patient has a "release date", the date they arrive in the system. Think of it as a preferred day to start the treatment, it may be the day when the patient gets to the city, or when he checks in at the hospital. Each patient also has a "wait time target", which is a tolerable number of days, counting from his release date, that the patient can wait until his treatment starts. Each day after the tolerance is penalized by a "wait time penalty" value.

Moreover, patients are categorized according to priorities from 1 to 4, increasingly. Patients with different priorities may have different waiting time targets penalties.

Each day on the horizon is expected to have a demand of patients of each priority. For example, we may expect only 1 patient of priority 4 and 3 patients of priority 3 to have a release date on day 1. Furthermore, there can be a list of patients waiting at the beginning of the time horizon, like a backlog. They may have already started treatment in the previous time horizon but haven't completed it, or may have not started treatment yet. Each of these waiting patients comes with their priority numbers, and also with the number of days they have been waiting (0 if treatment has already started). At the end of the horizon, we're expected to deliver an updated list of waiting patients, if any, to the next horizon.

Finally, we have more available data such as number of machines, regular hours capacity, overtime capacity (if needed), overtime cost (for each hour beyond regular hours), number of priorities, length of the time horizon, etc. The objective is to minimize overtime cost and waiting time penalty, which occurs only when patients start treatment late, with the priorities and number of delayed days as weights.


## Formulation
This formulation is inspired by Sana's formulation and contains some modifications.

### Data
- $I$ set of patients
- $J$ set of machines
- $T$ set of days in the plannig horizon


- $ru_{jt}$ = regular capacity (number of slots) of machine $j$ on day $t$.
- $ou_{jt}$ = overtime capacity (number of slots) of machine $j$ on day $t$.
- $oc_{jt}$ = overtime cost of machine $j$ on day $t$ (per slot).
- $pn_i$: priority number of patient $i$.
- $ns_i$: number of sessions of patient $i$.
- $sd_{ik}$: duration (number os slots) of session number $k$ of patient $i$.
- $wt_i$: wait time target of patient $i$.
- $rt_i$: release time (day) of patient $i$.
- $p_i$: wait time penalty of patient $i$.
- $h$: number of days in the horizon

### Decision variables
<!-- Formulation 1 (Luiz, Eric, ??):
- $x_{it} == 1$ if patient $i$ starts treatment on day $t$; $0$ otherwise.
- $z_{ijk} == 1$ if $k$-th session of patient $i$ takes place in machine $j$; $0$ otherwise.
- $y_{jt}$ number of extra hours of machine $j$ on day $t$. Non-negative integer. -->

- $x_{ijkt}$ equals $1$ if session $k$ of patient $i$ takes place on day $t$ in machine $j$, $0$ otherwise.
- $y_{jt}$ number of extra hours of machine $j$ on day $t$.

### Constraints

- _Each session must take place in at most one machine and in at most one day:_
$$
\sum_{jt} x_{ijkt} \leq 1, \quad \forall i, k.
$$
    Setting this constraint as an equation can easily turn the problem infeasible because there may not be enough capacity to serve all patients within the given time horizon.

- _At most one session per day:_
$$
\sum_{jk} x_{ijkt} \leq 1 , \quad \forall i,t.
$$
<!--    The $j$ index is completely useless in this constraint. If it improves performance, we can take off $j$ from the summation and put it together with $i$ and $t$. -->

- _Sessions occur on successive days:_
$$
\sum_{j} x_{ijkt} \leq \sum_j x_{ij,k+1,t+1}, \quad \forall i, k, t \; \text{such that}\; k+1 \leq ns_{i} \; \text{and}\; t+1 \leq h
$$
Alterantive formulation (possibly more efficient):
$$
\sum_{j} x_{ij1t} \leq \sum_j x_{ijk,t+k}, \quad \forall i, t, k \leq ns_{i}.
$$


- _Max number of extra hours_:
$$y_{jt} \leq eu_{jt}, \quad \forall j, t.$$

- _No machine can exceed its daily capacity:_
$$
\sum_{ik} sd_{ik} \cdot x_{ijkt} \leq ru_{jt} + y_{jt}, \quad \forall j, t
$$

- _No patient can start treatment before his release date:_
$$
x_{ijkt} = 0, \quad\forall i, j, k, t \; \text{such that}\; t \leq rt_i-1
$$
To be more efficient, we can simply not declare these variables in the implementation of the model.

**Todo:** Somehow, we must deal with patients registered in the previous horizon. If some patient has already started treatment previously, then we must guarantee he finishes treatment on the first days of current horizon; if he hasn't started treatment yet, so he may enter in our current horizon's list of patients, with value $0$ for release date (ie, like his release day were the first day on horizon). Remember: we will have already paid his partial penalty in the previous horizon. 

### Objective function

Our objective is to minimize both the sum of wait time penalties and the sum of overtimes cost.

- _Wait time penalties:_
\begin{align*}
\text{For each patient}\; i: \quad &L_i = \sum_{jt}(h-t)x_{ij1t}\\
& Delay_i = h-L_i-(rt_i+wt_i)\\[1.2ex]
& Total\_Penalty = \sum_{i} pn_i \cdot p_i \cdot Delay_i
\end{align*}
    
    Let's divide the interpretation into two cases:
    1. _Patient $i$ doesn't start treatment on this horizon:_ in this case, all $x_{ij1t}=0$, and therefore $L_i=0$. Then, $Delay_i=h-(rt_i+wt_i)$, which corresponds to the number of days (in this horizon) after the tolerance on the release date.
    2. _Patient $i$ starts treatment on this horizon:_ then, there exists exactly one machine $j$ and day $t$ for which $x_{ij1t} = 1$, $L_i=h-t$ and $Delay_i=t-(rt_i+wt_i)$, which is the number of days, after the tolerance, untill the patient starts treatment. 
    
- _Overtime cost:_
$$
Overtime\_Cost=\sum_{jt} oc_{jt} * y_{jt}
$$


Finally, we define $Objective = Total\_Penalty + Overtime\_Cost$

**Todo:** Add something about those patients that will not start treatment on this horizon. Their partial penalties must be paid right now, otherwise solver would prefer just to not arrange the patients on the horizon, and we want it to really try to arrange all patients now.